[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdgordillob/aca_aci_collab/blob/main/notebooks/aca_opt_full_replication.ipynb)

# aca_opt full replication: national temperature anomalies, 1961-2024

Runs the **actual** pipeline (not a 1-year smoke test) end-to-end in Colab: fetches the full raw ERA5 temperature archive from Drive, runs Stage 1 (merge/resample) over the real 1961-1990 baseline, Stage 2 (baseline percentiles) over that same 30-year window, and Stage 3 (anomalies) over the full 1961-2024 record -- then diffs the result against the official `salidas_colombia` output already on Drive to confirm this reproduces the real thing, not just a smoke test.

**Expect this to take roughly 45-90 minutes total** (network fetch of ~5.5GB across ~64 grib files, plus ~50-60 minutes of compute -- see `aca_opt_benchmark.ipynb`'s per-year timings). Keep the browser tab active; Colab disconnects idle sessions.

Prerequisite: `aca_opt_benchmark.ipynb` has already been run successfully at least once (confirms Drive access, dependencies, and `drive_sync.py` all work) -- this notebook doesn't re-explain those steps in as much detail.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_COLAB

## 1. Get the code

In [ ]:
import sys, os

REPO_ROOT = "/content/aca_indice_climatico_opt"

if IN_COLAB:
    !git clone --depth 1 https://github.com/mdgordillob/aca_indice_climatico_opt.git {REPO_ROOT}
else:
    REPO_ROOT = os.path.abspath("../../aca_indice_climatico_opt-main")  # adjust if running locally

sys.path.insert(0, os.path.join(REPO_ROOT, "src", "scripts"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src", "utils"))
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))  # for drive_sync

## 2. Install dependencies (pinned to requirements.txt)

See `aca_opt_benchmark.ipynb` for why these are pinned rather than left to resolve freely -- an unpinned `xarray` silently dropped output columns in an earlier run.

In [ ]:
if IN_COLAB:
    !apt-get -qq install -y libeccodes-dev > /dev/null
    !pip install -q cfgrib==0.9.14.1 eccodes==2.39.1 rioxarray==0.18.2 geopandas==1.0.1 netCDF4==1.7.2 xarray==2024.11.0

## 3. Mount Drive and fetch everything needed

Fetches, from `2. Datos`: every `era5_tmp_<year>.grib` under `era5/completos/` (not the whole folder -- that also has rain/wind grib this run doesn't need), `shapefiles/`, and the official `salidas_colombia.zip` (extracted into a *separate* path, `_official_reference/`, so it can't collide with -- or get overwritten by -- the output this notebook computes fresh).

In [ ]:
import shutil, time

DRIVE_ROOT = "/content/drive/MyDrive/2. Datos"
YEARS = range(1961, 2025)  # 1961-2024, matching the paper's stated coverage

raw_dir = os.path.join(REPO_ROOT, "data", "raw", "era5")
os.makedirs(raw_dir, exist_ok=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    import drive_sync
    drive_sync.sync(DRIVE_ROOT, REPO_ROOT, only=["shapefiles"])
    drive_sync.sync(
        DRIVE_ROOT, REPO_ROOT,
        mapping={"salidas_colombia": "data/processed/_official_reference/anomalias_colombia"},
    )

    completos = os.path.join(DRIVE_ROOT, "era5", "completos")
    t0 = time.perf_counter()
    fetched = 0
    for year in YEARS:
        name = f"era5_tmp_{year}.grib"
        src = os.path.join(completos, name)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(raw_dir, name))
            fetched += 1
    print(f"fetched {fetched}/{len(YEARS)} years of raw temperature grib in {time.perf_counter()-t0:.0f}s")
else:
    print("Not in Colab -- assuming data/raw/era5, data/shapefiles, and the official reference are already populated locally.")

## 4. Stage 1 -- merge/resample the 1961-1990 baseline

Only the baseline years are processed here: the real local `era5_daily_combined_tmp.nc` this mirrors spans exactly 1960-12-31 to 1990-12-31 (confirmed against this repo's own data), because Stage 2 only ever reads the `1961-1990` slice. Stage 3 (below) reads raw grib directly, independent of Stage 1's output, so it separately covers the full 1961-2024 range.

In [ ]:
import unir_archivos
from multiprocessing import Pool, cpu_count

stage1_out = os.path.join(REPO_ROOT, "data", "processed", "_full_run_daily_tmp")
os.makedirs(stage1_out, exist_ok=True)
baseline_years = [y for y in range(1961, 1991) if os.path.exists(os.path.join(raw_dir, f"era5_tmp_{y}.grib"))]
print(f"{len(baseline_years)}/30 baseline years present")

def _stage1_task_tmp(args):
    grib_path, year, out_dir = args
    import unir_archivos as u
    u.process_yearly_data_tmp(grib_path, year, "t2m", out_dir)

t0 = time.perf_counter()
tasks = [(os.path.join(raw_dir, f"era5_tmp_{y}.grib"), y, stage1_out) for y in baseline_years]
with Pool(max(1, cpu_count() - 1)) as pool:
    pool.map(_stage1_task_tmp, tasks)
unir_archivos.merge_yearly_files(
    stage1_out,
    os.path.join(stage1_out, "era5_daily_combined_tmp.nc"),
    "t2m",
)
stage1_time = time.perf_counter() - t0
print(f"Stage 1 ({len(baseline_years)} years, parallel): {stage1_time/60:.1f} min")

## 5. Stage 2 -- real 30-year baseline percentiles

In [ ]:
import calcular_percentil_temperatura as percentil_tmp

merged_file = os.path.join(stage1_out, "era5_daily_combined_tmp.nc")

t0 = time.perf_counter()
estadisticas = percentil_tmp.calcular_percentiles(merged_file)
percentil_tmp.guardar_percentiles(
    estadisticas,
    os.path.join(stage1_out, "era5_temperatura_percentil.nc"),
    stage1_out,
    guardar_csv=False,
)
stage2_time = time.perf_counter() - t0
print(f"Stage 2: {stage2_time/60:.1f} min")

## 6. Stage 3 -- anomalies, full 1961-2024

`use_multiprocessing=True` here (unlike the single-year smoke test) -- worth it across ~64 years, and Colab is Linux so the `sys.platform != 'win32'` guard the original script uses will enable it.

In [ ]:
import calcular_anomalias_temperatura as anomalias_tmp

stage3_out = os.path.join(REPO_ROOT, "data", "processed", "anomalias_colombia")
os.makedirs(stage3_out, exist_ok=True)
shapefile_path = os.path.join(REPO_ROOT, "data", "shapefiles", "colombia_4326.shp")
output_csv_path = os.path.join(stage3_out, "anomalies_temperature_combined.csv")

t0 = time.perf_counter()
anomalias_tmp.procesar_anomalias_temperatura(
    archivo_percentiles=os.path.join(stage1_out, "era5_temperatura_percentil.nc"),
    archivo_comparar_location=raw_dir,
    output_csv_path=output_csv_path,
    shapefile_path=shapefile_path if os.path.exists(shapefile_path) else None,
    output_netcdf=stage3_out,
    use_multiprocessing=True,
)
stage3_time = time.perf_counter() - t0
print(f"Stage 3: {stage3_time/60:.1f} min")

## 7. Compare against the official `salidas_colombia` output

Loads the freshly-computed CSV from Step 6 and the official reference fetched in Step 3, aligns on `(year, month)`, and reports the largest absolute difference per column. Small floating-point differences are expected (different hardware/BLAS); a `KeyError` or large systematic differences would mean something real changed --  check dependency versions first (Section 9.4 of `ARCHITECTURE.pdf` covers a real case of exactly that).

In [ ]:
import pandas as pd

official_path = os.path.join(
    REPO_ROOT, "data", "processed", "_official_reference",
    "anomalias_colombia", "anomalies_temperature_combined.csv",
)

mine = pd.read_csv(output_csv_path)
official = pd.read_csv(official_path)

compare_cols = [c for c in ["t_90", "t_10", "count_hot", "count_cold"] if c in mine.columns and c in official.columns]
missing_cols = [c for c in ["t_90", "t_10", "count_hot", "count_cold"] if c not in compare_cols]
if missing_cols:
    print(f"WARNING: columns missing from one side and skipped: {missing_cols}")

merged = mine.merge(official, on=["year", "month"], suffixes=("_mine", "_official"))
print(f"{len(merged)}/{len(official)} official rows matched by (year, month)")

for col in compare_cols:
    diff = (merged[f"{col}_mine"] - merged[f"{col}_official"]).abs()
    print(f"{col}: max abs diff = {diff.max():.2e}, mean abs diff = {diff.mean():.2e}")

## 8. Summary

In [ ]:
total_min = (stage1_time + stage2_time + stage3_time) / 60
print(f"Stage 1: {stage1_time/60:.1f} min")
print(f"Stage 2: {stage2_time/60:.1f} min")
print(f"Stage 3: {stage3_time/60:.1f} min")
print(f"Total compute: {total_min:.1f} min")